# Evaluasi Model Segmentasi Banjir

Notebook ini membandingkan U-Net dan Attention U-Net pada gambar uji yang sama. Evaluasi mencakup preprocessing, visualisasi mask, waktu inferensi, ukuran model, persentase area banjir, dan confidence.

Metrik Dice dan IoU memerlukan ground truth mask. Metrik tersebut tidak dihitung pada notebook ini jika ground truth belum tersedia.

In [ ]:
from pathlib import Path
import os
import time
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import load_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_HEIGHT = 256
IMG_WIDTH = 256
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

UNET_PATH = ROOT / 'model' / 'unet_flood_model_final.h5'
ATTENTION_PATH = ROOT / 'model' / 'attention_unet_flood.h5'
# Ganti dengan path gambar uji, contohnya: ROOT / 'Dataset' / 'nama_file.jpg'
TEST_IMAGE_PATH = ''

## Preprocessing

Semua model menerima input dengan pipeline yang sama: BGR ke RGB, resize menjadi 256 x 256, normalisasi 0-1, dan penambahan dimensi batch.

In [ ]:
def preprocess_image(path):
    path = Path(path)
    image = cv2.imread(str(path))
    if image is None:
        raise ValueError(f'Gambar tidak dapat dibaca: {path}')
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(image, (IMG_WIDTH, IMG_HEIGHT), interpolation=cv2.INTER_AREA)
    normalized = resized.astype(np.float32) / 255.0
    return image, np.expand_dims(normalized, axis=0)

def postprocess_mask(prediction, threshold=0.5):
    mask = prediction[0, :, :, 0]
    return (mask > threshold).astype(np.uint8) * 255

def prediction_metrics(prediction):
    mask = prediction[0, :, :, 0]
    binary = mask > 0.5
    flood_pixels = int(binary.sum())
    return {
        'flood_percentage': float(binary.mean() * 100),
        'confidence': float(mask[binary].mean() * 100) if flood_pixels else 0.0,
    }

In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true = K.flatten(y_true)
    y_pred = K.flatten(y_pred)
    intersection = K.sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (K.sum(y_true) + K.sum(y_pred) + smooth)

def load_segmentation_model(path):
    if not path.exists():
        raise FileNotFoundError(f'Model tidak ditemukan: {path}')
    return load_model(str(path), custom_objects={'dice_coef': dice_coef}, compile=False)

models = {
    'U-Net': load_segmentation_model(UNET_PATH),
    'Attention U-Net': load_segmentation_model(ATTENTION_PATH),
}
print('Model berhasil dimuat.')

In [ ]:
if not TEST_IMAGE_PATH:
    print('Isi TEST_IMAGE_PATH dengan path gambar uji, lalu jalankan ulang sel ini.')
else:
    original_image, input_tensor = preprocess_image(TEST_IMAGE_PATH)
    print(f'Gambar uji: {TEST_IMAGE_PATH}')
    print(f'Ukuran input model: {input_tensor.shape}, tipe data: {input_tensor.dtype}')

In [ ]:
if TEST_IMAGE_PATH:
    results = {}
    for name, model in models.items():
        start = time.perf_counter()
        prediction = model.predict(input_tensor, verbose=0)
        elapsed = time.perf_counter() - start
        results[name] = {
            'prediction': prediction,
            'mask': postprocess_mask(prediction),
            'inference_seconds': elapsed,
            **prediction_metrics(prediction),
        }

    figure, axes = plt.subplots(1, len(results) + 1, figsize=(15, 4))
    axes[0].imshow(original_image)
    axes[0].set_title('Gambar asli')
    axes[0].axis('off')
    for axis, (name, result) in zip(axes[1:], results.items()):
        axis.imshow(result['mask'], cmap='gray')
        axis.set_title(name)
        axis.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
if TEST_IMAGE_PATH:
    comparison = []
    for name, result in results.items():
        model_path = UNET_PATH if name == 'U-Net' else ATTENTION_PATH
        comparison.append({
            'model': name,
            'model_size_mb': round(model_path.stat().st_size / (1024 ** 2), 2),
            'parameters': models[name].count_params(),
            'inference_seconds': round(result['inference_seconds'], 4),
            'flood_percentage': round(result['flood_percentage'], 2),
            'confidence': round(result['confidence'], 2),
        })
    comparison_df = pd.DataFrame(comparison)
    display(comparison_df)
else:
    print('Tabel perbandingan akan muncul setelah TEST_IMAGE_PATH diisi.')

## Catatan metodologi jurnal

- Gunakan gambar uji dan preprocessing yang sama untuk kedua model.
- Ukur waktu inferensi pada hardware dan kondisi yang sama, lalu ulangi beberapa kali dan laporkan rata-rata.
- Metrik `flood_percentage` dan `confidence` di notebook ini adalah metrik keluaran prediksi, bukan ukuran akurasi terhadap label.
- Untuk menghitung Dice dan IoU, sediakan ground truth mask dengan ukuran dan pasangan file yang benar.
- Catat versi Python, TensorFlow, OpenCV, hardware, seed, dan pembagian dataset agar eksperimen dapat direproduksi.